Plotting 

In [ ]:
# violin plots

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

mobility = pd.read_csv("data/mobility_clean.csv")
mobility['date'] = pd.to_datetime(mobility['date'])

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 12))

sns.set_style("darkgrid")

sns.violinplot(y=mobility["retail_and_recreation_percent_change_from_baseline"], ax=axes[0, 0])
sns.violinplot(y=mobility["grocery_and_pharmacy_percent_change_from_baseline"], ax=axes[0, 1])
sns.violinplot(y=mobility["transit_stations_percent_change_from_baseline"], ax=axes[1, 2])
sns.violinplot(y=mobility["workplaces_percent_change_from_baseline"], ax=axes[0, 2])
sns.violinplot(y=mobility["residential_percent_change_from_baseline"], ax=axes[1, 0])

for ax in [axes[0, 0], axes[1, 0]]: 
    ax.set_ylabel("Percent Change (%)")

for ax in [axes[0, 1], axes[0, 2]]: 
    ax.set_ylabel('')
    ax.set_yticklabels([])

for ax in axes.flatten(): 
    ax.set_ylim(-130, 130)

axes[1, 2].set_ylim(-300, 300)
axes[1, 2].set_ylabel('')

axes[1, 0].set_ylim(-50, 50)

axes[0, 0].set_title("Retail and Recreation")
axes[0, 1].set_title("Grocery and Pharmacy")
axes[1, 2].set_title("Transit Stations")
axes[0, 2].set_title("Workplaces")
axes[1, 0].set_title("Residential")

fig.suptitle("Violin Plots of Mobility by Category", fontsize=20)

fig.delaxes(axes[1, 1])


In [ ]:
# overall cat trends

mobility_cats = mobility.copy()
mobility_cats["year_month"] = mobility_cats["date"].dt.to_period('M')


cat_cols = [
    "retail_and_recreation_percent_change_from_baseline", 
    "grocery_and_pharmacy_percent_change_from_baseline", 
    "transit_stations_percent_change_from_baseline", 
    "workplaces_percent_change_from_baseline", 
    "residential_percent_change_from_baseline"
]
monthly_mob = mobility_cats.groupby("year_month")[cat_cols].mean()
monthly_mob.index = monthly_mob.index.to_timestamp()

monthly_mob_long = (
    monthly_mob
    .reset_index()
    .melt(id_vars="year_month", value_vars=cat_cols,
          var_name="category", value_name="percent_change")
)

monthly_mob_long["category_short"] = monthly_mob_long["category"].replace({
    "retail_and_recreation_percent_change_from_baseline":"retail/recreation", 
    "grocery_and_pharmacy_percent_change_from_baseline":"grocery/pharmacy", 
    "transit_stations_percent_change_from_baseline":"transit", 
    "workplaces_percent_change_from_baseline":"workplaces", 
    "residential_percent_change_from_baseline":"residential"
    }
)

palette = sns.crayon_palette(["Cerulean", "Atomic Tangerine", "Brick Red", "Purple Mountains' Majesty", "Pine Green"])

fig, ax = plt.subplots(figsize=(12, 7))
sns.lineplot(
    data=monthly_mob_long, 
    x="year_month", 
    y="percent_change", 
    hue="category_short", palette=palette
)

sns.set_style(style="darkgrid")
ax.legend(title="Category", loc="upper right", fontsize=13, title_fontsize=15)


ax.set_title(" Monthly Percent Change from Baseline", fontsize=20)
ax.set_xlabel("Month", fontsize=15)
ax.set_ylabel("Percent Change (%)")
ax.grid(True) 
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.xticks(rotation=45, fontsize=15, style='italic')

plt.ylim(-50, 50)
plt.tight_layout() 
plt.show()


In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 12))

sns.set_style("darkgrid")
sns.violinplot(y=monthly_mob["retail_and_recreation_percent_change_from_baseline"], ax=axes[0, 0])
sns.violinplot(y=monthly_mob["grocery_and_pharmacy_percent_change_from_baseline"], ax=axes[0, 1])
sns.violinplot(y=monthly_mob["transit_stations_percent_change_from_baseline"], ax=axes[1, 2])
sns.violinplot(y=monthly_mob["workplaces_percent_change_from_baseline"], ax=axes[0, 2])
sns.violinplot(y=monthly_mob["residential_percent_change_from_baseline"], ax=axes[1, 0])

for ax in [axes[0, 0], axes[1, 0]]: 
    ax.set_ylabel("Percent Change (%)")

for ax in [axes[0, 1], axes[0, 2], axes[1, 2]]: 
    ax.set_ylabel('')
    ax.set_yticklabels([])

for ax in axes.flatten(): 
    ax.set_ylim(-50, 50)

axes[0, 0].set_title("Retail and Recreation", fontsize=16)
axes[0, 1].set_title("Grocery and Pharmacy", fontsize=16)
axes[1, 2].set_title("Transit Stations", fontsize=16)
axes[0, 2].set_title("Workplaces", fontsize=16)
axes[1, 0].set_title("Residential", fontsize=16)

fig.suptitle("Violin Plots of Monthly Mobility by Category", fontsize=20)

fig.delaxes(axes[1, 1])

In [ ]:
# rural v urban
rural_counties = ["Accomack County","Alleghany County","Bath County",
                  "Bland County","Brunswick County","Buchanan County",
                  "Carroll County","Charlotte County","Craig County",
                  "Dickenson County","Essex County","Grayson County",
                  "Greensville County","Halifax County","Henry County",
                  "Highland County","Lee County","Louisa County",
                  "Lunenburg County","Madison County","Mecklenburg County",
                  "Middlesex County","Montgomery County","Nelson County",
                  "Northampton County","Northumberland County","Patrick County",
                  "Pittsylvania County","Prince Edward County","Pulaski County",
                  "Richmond County","Rockbridge County","Rockingham County",
                  "Russell County","Smyth County","Southampton County",
                  "Tazewell County","Wise County","Wythe County","Shenandoah County"]

rural_mobility = mobility[mobility["sub_region_2"].isin(rural_counties)].copy()
metro_mobility = mobility[-mobility["sub_region_2"].isin(rural_counties)].copy()

# rural
rural_mobility.loc[:,"year_month"] = rural_mobility["date"].dt.to_period('M')

monthly_mob_rural = rural_mobility.groupby("year_month")[cat_cols].mean()
monthly_mob_rural = monthly_mob_rural.sort_index()

# metro
metro_mobility.loc[:,"year_month"] = metro_mobility["date"].dt.to_period('M')

monthly_mob_metro = metro_mobility.groupby("year_month")[cat_cols].mean()
monthly_mob_metro = monthly_mob_metro.sort_index()
monthly_mob_metro.index = monthly_mob_metro.index.to_timestamp()




In [ ]:
print(rural_mobility.shape)
print(mobility.shape)
print(metro_mobility.shape)



In [ ]:
import seaborn as sns


monthly_r = (
    monthly_mob_rural
    .reset_index()
    .melt(id_vars="year_month", value_vars=cat_cols,
          var_name="category", value_name="percent_change")
)
monthly_r['year_month'] = monthly_r['year_month'].dt.strftime("%Y-%m")

monthly_r["category_short"] = monthly_r["category"].replace({
    "retail_and_recreation_percent_change_from_baseline":"retail/recreation", 
    "grocery_and_pharmacy_percent_change_from_baseline":"grocery/pharmacy",  
    "transit_stations_percent_change_from_baseline":"transit", 
    "workplaces_percent_change_from_baseline":"workplaces", 
    "residential_percent_change_from_baseline":"residential"
    }
)


palette = sns.crayon_palette(["Cerulean", "Atomic Tangerine", "Brick Red", "Purple Mountains' Majesty", "Pine Green"])

fig, ax = plt.subplots(figsize=(12, 7))
sns.set_style(style='darkgrid')
sns.lineplot(data=monthly_r, x="year_month", y="percent_change", hue="category_short", palette=palette)
plt.ylim(-50, 50)

plt.title("Monthly Percent Change from Baseline: Non-Metro", fontsize=20)
plt.xlabel("Month", fontsize=15)
plt.ylabel("Percent Change (%)", fontsize=15)
plt.grid(True)
plt.axhline(0, linestyle='--', linewidth=0.8)
ax.legend(title="Category", loc="upper left", fontsize=13, title_fontsize=15)

plt.xticks(rotation=45, fontsize=15, style="italic")
plt.yticks(style="italic")
ax.set_xticks(monthly_r["year_month"].unique()[::3])

plt.tight_layout()
plt.show()

In [ ]:

monthly_m = (
    monthly_mob_metro
    .reset_index()
    .melt(id_vars="year_month", value_vars=cat_cols,
          var_name="category", value_name="percent_change")
)

monthly_m["category_short"] = monthly_m["category"].replace({
    "retail_and_recreation_percent_change_from_baseline":"retail/recreation", 
    "grocery_and_pharmacy_percent_change_from_baseline":"grocery/pharmacy",  
    "transit_stations_percent_change_from_baseline":"transit", 
    "workplaces_percent_change_from_baseline":"workplaces", 
    "residential_percent_change_from_baseline":"residential"
    }
)

palette = sns.crayon_palette(["Cerulean", "Atomic Tangerine", "Brick Red", "Purple Mountains' Majesty", "Pine Green"])
monthly_m['year_month'] = monthly_m['year_month'].dt.strftime("%Y-%m")

fig, ax = plt.subplots(figsize=(12, 7))
sns.set_style(style="darkgrid")
sns.lineplot(data=monthly_m, x="year_month", y="percent_change", hue="category_short", palette=palette)
plt.ylim(-50, 50)

plt.title("Monthly Percent Change from Baseline: Metro", fontsize=20)
plt.xlabel("Month", fontsize=15)
plt.ylabel("Percent Change (%)", fontsize=15)
plt.grid(True)
plt.axhline(0, linestyle='--', linewidth=0.8)
ax.legend(title="Category", loc="upper right", fontsize=13, title_fontsize=15)

plt.xticks(rotation=45, fontsize=15, style="italic")
plt.yticks(style="italic")
ax.set_xticks(monthly_m["year_month"].unique()[::3])


plt.tight_layout()
plt.show()

In [ ]:
list(sns.palettes.crayons.keys())
